# TP 1 — Premières transformations DataFrame
**Durée indicative : 40–55 min**

Objectifs : créer un DataFrame, renommer et ajouter une colonne, filtrer et agréger, écrire des partitions Parquet.

> **Environment:** Databricks notebook in a Git folder. Attach a compute resource with PySpark available. Data files are in the adjacent `data/` folder. This lab uses only **synthetic** data.

> **Expected skill level:** first experience with Big Data / Spark. Follow the cells in order.


In [1]:
from pyspark.sql import functions as F
data = [("James", "Smith", "USA", 1, 200), ("Michael", "Rose", "FR", 2, 4000), ("Robert", "Williams", "FR", 3, 50), ("Maria", "Jones", "USA", 4, 200)]
columns = ["Prenom", "Nom", "Pays", "ID", "value"]
df = spark.createDataFrame(data, columns)
df.show()
df.printSchema()

+-------+--------+----+---+-----+
| Prenom|     Nom|Pays| ID|value|
+-------+--------+----+---+-----+
|  James|   Smith| USA|  1|  200|
|Michael|    Rose|  FR|  2| 4000|
| Robert|Williams|  FR|  3|   50|
|  Maria|   Jones| USA|  4|  200|
+-------+--------+----+---+-----+

root
 |-- Prenom: string (nullable = true)
 |-- Nom: string (nullable = true)
 |-- Pays: string (nullable = true)
 |-- ID: long (nullable = true)
 |-- value: long (nullable = true)



## 1. Transformation sans modifier la source
`withColumn` produit un **nouveau** DataFrame. Vérifiez que `df` n’a pas changé.

In [2]:
df2 = df.withColumnRenamed("value", "montant")
df3 = df2.withColumn("date_reference", F.lit(None).cast("date"))
df3.show()
df.show()

+-------+--------+----+---+-------+--------------+
| Prenom|     Nom|Pays| ID|montant|date_reference|
+-------+--------+----+---+-------+--------------+
|  James|   Smith| USA|  1|    200|          NULL|
|Michael|    Rose|  FR|  2|   4000|          NULL|
| Robert|Williams|  FR|  3|     50|          NULL|
|  Maria|   Jones| USA|  4|    200|          NULL|
+-------+--------+----+---+-------+--------------+



+-------+--------+----+---+-----+
| Prenom|     Nom|Pays| ID|value|
+-------+--------+----+---+-----+
|  James|   Smith| USA|  1|  200|
|Michael|    Rose|  FR|  2| 4000|
| Robert|Williams|  FR|  3|   50|
|  Maria|   Jones| USA|  4|  200|
+-------+--------+----+---+-----+



In [3]:
# Complément : vérification explicite, df n'a pas été modifié
print("Colonnes de df  :", df.columns)
print("Colonnes de df3 :", df3.columns)

Colonnes de df  : ['Prenom', 'Nom', 'Pays', 'ID', 'value']
Colonnes de df3 : ['Prenom', 'Nom', 'Pays', 'ID', 'montant', 'date_reference']


**✅ Réponse**

`df` est **inchangé** : il a toujours sa colonne `value` et pas de `date_reference`. Les DataFrames Spark sont **immuables** : `withColumnRenamed` et `withColumn` ne modifient pas `df`, elles renvoient un **nouveau** DataFrame (`df2`, puis `df3`). C'est pour cela qu'on range le résultat dans une nouvelle variable.

Ces transformations sont aussi **paresseuses** (*lazy*) : Spark ne calcule rien tant qu'une **action** (`show`, `count`, `write`…) n'est pas demandée, il construit seulement un plan d'exécution.

## 2. NULL est-il égal au texte "NULL" ?
On utilise `None` pour une vraie valeur nulle. Remplacez-la par une date arbitraire uniquement pour l’exercice.

In [4]:
df4 = df3.withColumn("date_reference", F.coalesce(F.col("date_reference"), F.lit("2026-01-01").cast("date")))
df4.show()

+-------+--------+----+---+-------+--------------+
| Prenom|     Nom|Pays| ID|montant|date_reference|
+-------+--------+----+---+-------+--------------+
|  James|   Smith| USA|  1|    200|    2026-01-01|
|Michael|    Rose|  FR|  2|   4000|    2026-01-01|
| Robert|Williams|  FR|  3|     50|    2026-01-01|
|  Maria|   Jones| USA|  4|    200|    2026-01-01|
+-------+--------+----+---+-------+--------------+



In [5]:
# Complément : un vrai NULL n'est pas le texte "NULL"
spark.sql("""
    SELECT NULL IS NULL    AS null_est_null,
           'NULL' IS NULL  AS texte_est_null,
           NULL = NULL     AS null_egal_null,
           NULL <=> NULL   AS egalite_null_safe,
           length('NULL')  AS longueur_du_texte
""").show()

# count(colonne) ignore les NULL, count(*) compte toutes les lignes
df3.select(F.count("*").alias("nb_lignes"), F.count("date_reference").alias("nb_dates_renseignees")).show()

+-------------+--------------+--------------+-----------------+-----------------+
|null_est_null|texte_est_null|null_egal_null|egalite_null_safe|longueur_du_texte|
+-------------+--------------+--------------+-----------------+-----------------+
|         true|         false|          NULL|             true|                4|
+-------------+--------------+--------------+-----------------+-----------------+



+---------+--------------------+
|nb_lignes|nb_dates_renseignees|
+---------+--------------------+
|        4|                   0|
+---------+--------------------+



**✅ Réponse — NULL est-il égal au texte "NULL" ?**

**Non.** Un vrai `NULL` (créé avec `None` ou `F.lit(None)`) signifie « valeur absente, inconnue ». Le texte `"NULL"` est une chaîne ordinaire de 4 caractères, donc une vraie valeur. Les sorties ci-dessus le montrent :
- `IS NULL` (ou `.isNull()`) est vrai pour un `NULL`, faux pour `'NULL'` ;
- `NULL = NULL` ne vaut pas `true` mais `NULL` : Spark suit une logique à trois valeurs (vrai, faux, inconnu) et ne peut pas dire si deux inconnues sont égales. Pour tester l'absence de valeur on utilise `IS NULL`, ou `<=>` (`eqNullSafe`) pour comparer en tenant compte des NULL ;
- `count("date_reference")` vaut 0 : les agrégats ignorent les `NULL`. Un texte `"NULL"` aurait été compté 4 fois ;
- `coalesce` ne remplace que les vrais `NULL` : il a bien mis `2026-01-01`. Avec le texte `"NULL"` (le piège du TP d'origine, qui utilisait `lit("NULL")`), rien n'aurait été remplacé et la colonne aurait été de type `string` au lieu de `date`.

D'où `F.lit(None).cast("date")` : un vrai NULL, **typé** `date`.

## 3. Filtrer puis agréger
Affichez la ligne de l’ID 4. Puis comptez les personnes et additionnez les montants **par pays**.

In [6]:
df4.filter(F.col("ID") == 4).show()
df4.groupBy("Pays").agg(F.count("*").alias("nb_personnes"), F.sum("montant").alias("total_montant")).show()

+------+-----+----+---+-------+--------------+
|Prenom|  Nom|Pays| ID|montant|date_reference|
+------+-----+----+---+-------+--------------+
| Maria|Jones| USA|  4|    200|    2026-01-01|
+------+-----+----+---+-------+--------------+



+----+------------+-------------+
|Pays|nb_personnes|total_montant|
+----+------------+-------------+
|  FR|           2|         4050|
| USA|           2|          400|
+----+------------+-------------+



**✅ Réponse**

- La ligne d'`ID` 4 est **Maria Jones (USA), montant 200**.
- Par pays : **FR** → 2 personnes, total **4 050** (4 000 + 50) ; **USA** → 2 personnes, total **400** (200 + 200).

`filter` sélectionne des lignes ; `groupBy(...).agg(...)` produit **une ligne par pays** avec les agrégats demandés (`count`, `sum`), renommés avec `alias`.

## 4. Écrire du Parquet partitionné
Quelles sous-arborescences `Pays=...` attendez-vous ? Attention à la cardinalité : on ne partitionne pas par identifiant unique.

In [7]:
# IMPORTANT: configure one writable Unity Catalog Volume before this step.
# Use a unique subdirectory to avoid overwriting a classmate's output.
import os, re
from pathlib import Path

# Identifiant personnel déduit du compte connecté (ex. "jean.dupont@episen.fr" -> "jean_dupont") :
# chaque étudiant écrit dans son propre sous-dossier, jamais à la racine du volume partagé.
IDENTIFIANT = re.sub(r"[^a-z0-9]+", "_", spark.sql("SELECT current_user()").first()[0].split("@")[0].lower()).strip("_")

if "DATABRICKS_RUNTIME_VERSION" in os.environ or str(Path.cwd()).startswith("/Workspace"):
    VOLUME = "/Volumes/workspace/default/episen_volume"
    assert os.path.isdir(VOLUME), f"Volume introuvable : {VOLUME} (le créer dans Catalog, ou adapter le chemin)"
    OUTPUT_ROOT = f"{VOLUME}/{IDENTIFIANT}"
else:
    # Exécution hors Databricks (Spark installé en local) : dossier output/ du projet, ignoré par Git
    OUTPUT_ROOT = str(Path.cwd().parent / "output" / IDENTIFIANT)

assert "CHANGE_ME" not in OUTPUT_ROOT, "Replace CHANGE_ME with your personal identifier."
print("Dossier de sortie :", OUTPUT_ROOT)

Dossier de sortie : /tmp/EPISEN_TP_1/output/etudiant


In [8]:
path = OUTPUT_ROOT.rstrip("/") + "/tp01_personnes_parquet"
df4.write.mode("overwrite").partitionBy("Pays").parquet(path)
spark.read.parquet(path).show()

+-------+--------+---+-------+--------------+----+
| Prenom|     Nom| ID|montant|date_reference|Pays|
+-------+--------+---+-------+--------------+----+
| Robert|Williams|  3|     50|    2026-01-01|  FR|
|Michael|    Rose|  2|   4000|    2026-01-01|  FR|
|  James|   Smith|  1|    200|    2026-01-01| USA|
|  Maria|   Jones|  4|    200|    2026-01-01| USA|
+-------+--------+---+-------+--------------+----+



In [9]:
# Complément : l'arborescence produite sur le disque
for dossier in sorted(p for p in Path(path).iterdir() if p.is_dir()):
    nb_fichiers = len([f for f in dossier.iterdir() if f.suffix == ".parquet"])
    print(f"{dossier.name:<10} -> {nb_fichiers} fichier(s) Parquet")

# Lecture filtrée sur la colonne de partition : regarder « PartitionFilters » dans le plan
spark.read.parquet(path).filter(F.col("Pays") == "FR").explain()

Pays=FR    -> 2 fichier(s) Parquet
Pays=USA   -> 2 fichier(s) Parquet
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [Prenom#184,Nom#185,ID#186L,montant#187L,date_reference#188,Pays#189] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/EPISEN_TP_1/output/etudiant/tp01_personnes_parquet], PartitionFilters: [isnotnull(Pays#189), (Pays#189 = FR)], PushedFilters: [], ReadSchema: struct<Prenom:string,Nom:string,ID:bigint,montant:bigint,date_reference:date>




**✅ Réponse — quelles sous-arborescences ?**

On attendait **un sous-dossier par valeur distincte** de `Pays`, soit `Pays=FR/` et `Pays=USA/` : c'est bien ce qu'on obtient. La valeur de `Pays` n'est plus stockée dans les fichiers, elle est encodée dans le **nom du dossier**. À la relecture, Spark reconstruit la colonne `Pays` à partir de ces noms et la place **en dernière position**.

Le plan affiche `PartitionFilters: [... (Pays = FR)]` : grâce au partitionnement, un filtre sur `Pays` ne lit que le dossier `Pays=FR` (*partition pruning*).

Cardinalité : partitionner par `ID` créerait **un dossier par personne**. Sur des millions de lignes, cela donnerait des millions de petits fichiers, très lents à lister et à lire. On partitionne par une colonne à **faible cardinalité**, souvent utilisée dans les filtres (pays, date…).

## 5. SQL ou API DataFrame ?
Les deux exécutent des traitements Spark. Comparez :

In [10]:
df4.createOrReplaceTempView("personnes")
spark.sql("SELECT Pays, COUNT(*) AS nb FROM personnes GROUP BY Pays").show()

+----+---+
|Pays| nb|
+----+---+
|  FR|  2|
| USA|  2|
+----+---+



In [11]:
# Complément : le même calcul avec l'API DataFrame, puis les deux plans d'exécution
df4.groupBy("Pays").agg(F.count("*").alias("nb")).show()

spark.sql("SELECT Pays, COUNT(*) AS nb FROM personnes GROUP BY Pays").explain()
df4.groupBy("Pays").agg(F.count("*").alias("nb")).explain()

+----+---+
|Pays| nb|
+----+---+
|  FR|  2|
| USA|  2|
+----+---+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Pays#2], functions=[count(1)])
   +- Exchange hashpartitioning(Pays#2, 200), ENSURE_REQUIREMENTS, [plan_id=310]
      +- HashAggregate(keys=[Pays#2], functions=[partial_count(1)])
         +- Project [Pays#2]
            +- Scan ExistingRDD[Prenom#0,Nom#1,Pays#2,ID#3L,value#4L]


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Pays#2], functions=[count(1)])
   +- Exchange hashpartitioning(Pays#2, 200), ENSURE_REQUIREMENTS, [plan_id=327]
      +- HashAggregate(keys=[Pays#2], functions=[partial_count(1)])
         +- Project [Pays#2]
            +- Scan ExistingRDD[Prenom#0,Nom#1,Pays#2,ID#3L,value#4L]




**✅ Réponse — SQL ou API DataFrame ?**

Même résultat (FR : 2, USA : 2) et **même plan physique** : les deux `explain()` affichent exactement les mêmes opérations. Sur Databricks, c'est une agrégation exécutée par le moteur Photon (`PhotonGroupingAgg(keys=[Pays], functions=[count(1)])`) ; avec Spark seul, on verrait `HashAggregate` → `Exchange` → `HashAggregate`. SQL et API DataFrame sont deux façons d'écrire le même traitement, que Spark traduit et optimise avec le même moteur (Catalyst) : **aucune différence de performance**.

Le choix dépend du contexte :
- **SQL** : déclaratif, connu de tous (analystes comme data engineers), pratique pour explorer ;
- **API DataFrame** : s'intègre au code Python (variables, fonctions réutilisables, boucles, tests).

On peut mélanger les deux grâce aux vues temporaires.

### Questions de bilan
- Quelle différence entre `filter` et `groupBy` ?
- Pourquoi un vrai `NULL` est-il différent de `"NULL"` ?
- Que gagne-t-on (ou perd-on) en partitionnant des données ?

**✅ Réponses du bilan**

**1. Quelle différence entre `filter` et `groupBy` ?**
- `filter` garde ou écarte **chaque ligne** selon une condition, indépendamment des autres : mêmes colonnes, moins de lignes. Chaque partition est traitée sur place, sans échange de données entre machines (transformation *étroite*).
- `groupBy(...).agg(...)` **regroupe** les lignes qui ont la même clé et calcule **une ligne par groupe** (nouvelles colonnes : `count`, `sum`…). Sur de vraies données réparties sur plusieurs machines, il faut réunir les lignes d'une même clé au même endroit : c'est un **shuffle** (opération `Exchange` dans un plan Spark), plus coûteux (transformation *large*). Sur nos 4 lignes, tout tient dans une seule partition : le plan Databricks ci-dessus n'en a même pas besoin.
- Bonne pratique : filtrer le plus tôt possible, avant d'agréger ou de joindre.

**2. Pourquoi un vrai `NULL` est-il différent de `"NULL"` ?**
Un vrai `NULL` est l'absence de valeur : détecté par `isNull()`, ignoré par les agrégats, remplaçable par `coalesce`, et compatible avec n'importe quel type (ici `date`). `"NULL"` est un texte de 4 caractères, une valeur comme une autre : il fausse les comptages, échappe aux tests de nullité et force la colonne en `string`.

**3. Que gagne-t-on (ou perd-on) en partitionnant ?**
- Gains :
  - *partition pruning* : un filtre sur la colonne de partition ne lit que les dossiers utiles (`PartitionFilters`), un gros gain sur de gros volumes filtrés par date ou par pays ;
  - une organisation lisible, et la possibilité de réécrire une seule partition (un jour, par exemple).
- Pertes et risques :
  - **trop de petits fichiers** si la colonne a une forte cardinalité (identifiant, horodatage) ou si les partitions sont petites : ici, 4 lignes ont déjà produit plusieurs fichiers ;
  - aucun gain pour les requêtes qui ne filtrent pas sur cette colonne, et un déséquilibre si une valeur concentre l'essentiel des données ;
  - la colonne sort des fichiers : son type est **deviné** à partir du nom de dossier à la relecture (un code `"01"` revient sous forme d'entier `1`).